# ***Hierarchical Clustering - (Clustering Jerárquico)***



## **Introducción a Hierarchical clustering**

Hierarchical clustering es una alternativa a los métodos de partitioning clustering (como K-means) que no requiere que se pre-especifique el número de clusters, la encontrará por sí mismo.

Además, permite el trazado de dendrogramas.

Los dendrogramas son visualizaciones de una agrupación jerárquica binaria.

### **Teoria**

Existen dos enfoques para este tipo de agrupación: aglomerativo y divisivo.

**Divisivo:** este método comienza por englobar todos los puntos de datos en un solo grupo. Luego, dividirá el grupo iterativamente en otros más pequeños hasta que cada uno de ellos contenga sólo una muestra.

**Aglomerativo:** este método comienza con cada muestra siendo un grupo diferente y luego fusionándolas por las que están más cerca unas de otras hasta que sólo haya un grupo.



### **AGLOMERATIVO: el algoritmo de este tipo es:**

1-. Considerar cada una de las n observaciones como un clúster individual, formando así la base del dendrograma (hojas).

2-. Proceso iterativo hasta que todas las observaciones pertenecen a un único cluster:

2-.1 Calcular la distancia entre cada posible par de los n clusters. El investigador debe determinar el tipo de medida empleada para cuantificar la similitud entre observaciones o grupos (distancia y linkage).

2-.2 Los dos clusters más similares se fusionan, de forma que quedan n-1 clusters.

Cortar la estructura de árbol generada (dendrograma) a una determinada altura para crear los clusters finales.

Para que el proceso de agrupamiento pueda llevarse a cabo tal como indica el algoritmo anterior, es necesario **definir cómo se cuantifica la similitud entre dos clusters.**

Es decir, se tiene que extender el concepto de distancia entre pares de observaciones para que sea aplicable a pares de grupos, cada uno formado por varias observaciones.

A este **proceso se le conoce como linkage**. A continuación, se describen los 5 tipos de linkage más empleados:


- Complete or Maximum
- Single or Minimum:
- Average: Centroid:
- Ward:





## **DIVISIVO : el algoritmo más conocido de divisive hierarchical clustering es DIANA**
(DIvisive ANAlysis Clustering).

1.- Todas las n observaciones forman un único cluster.

Repetir hasta que haya n clusters:

2.1- Calcular para cada cluster la mayor de las distancias entre pares de observaciones (diámetro del cluster).

2.2- Seleccionar el cluster con mayor diámetro.

2.3- Calcular la distancia media de cada observación respecto a las demás.

2.4- La observación más distante inicia un nuevo cluster.

2.5- Se reasignan las observaciones restantes al nuevo cluster o al viejo dependiendo de cuál está más próximo.



### **DENDROGRAMA**

Además de representar en un dendrograma la similitud entre observaciones, se tiene que identificar el número de clusters creados y qué observaciones forman parte de cada uno.

Si se realiza un corte horizontal a una determinada altura del dendrograma, el número de ramas que sobrepasan (en sentido ascendente) dicho corte se corresponde con el número de clusters.

La imagen muestra dos veces el mismo dendrograma.

Si se realiza el corte a la altura de 5, se obtienen dos clusters, mientras que si se hace a la de 3.5 se obtienen 4.

La altura de corte tiene por lo tanto la misma función que el valor K en K-means-clustering: controla el número de clusters obtenidos.

DENDROGRAMA

Dos propiedades adicionales se derivan de la forma en que se generan los clusters en el método de hierarchical clustering:

- Dada la longitud variable de las ramas, siempre existe un intervalo de altura para el que cualquier corte de lugar al mismo número de clusters. En el ejemplo, todos los cortes entre las alturas 5 y 6 tienen como resultado los mismos 2 clusters.

- Con un solo dendrograma se dispone de la flexibilidad para generar cualquier número de clusters desde 1 a n. La selección del número óptimo puede valorarse de forma visual, tratando de identificar las ramas principales en base a la altura a la que ocurren las uniones. En el ejemplo expuesto es razonable elegir entre 2 o 4 clusters.



### **Conceptos** (Se aplica en todos los clustering

**n_samples** En clustering, un sample significa simplemente:

- Una observación
- Un dato individual
- Una fila del dataset
- Un punto en el espacio de características


In [10]:
%load_ext kedro.ipython 
catalog.keys()
df_fifa = catalog.load("model_input_table")

The kedro.ipython extension is already loaded. To reload it, use:
  %reload_ext kedro.ipython


[03/01/26 04:42:53] INFO     Loading data from model_input_table (ParquetDataset)...           ]8;id=321409;file://C:\Users\brand\Downloads\Proyecto_ML_Kedro\.venv\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=125794;file://C:\Users\brand\Downloads\Proyecto_ML_Kedro\.venv\Lib\site-packages\kedro\io\data_catalog.py#1046\1046]8;;\

#### **Importaciónes**

In [11]:
# -- Tratamiento de datos --
import numpy as np
import pandas as pd

# -- Gráficos --
import seaborn as sns
import seaborn as sb


# -- Procesado y modelado --
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.preprocessing import StandardScaler

# Importo el método de clustering jerárquico (bottom-up)
from sklearn.cluster import AgglomerativeClustering

# Para graficar el Elbow Method
import matplotlib.pyplot as plt

# importamos el puntaje de silhouette
from sklearn.metrics import silhouette_score

# -- Dendrogramas --
from scipy.cluster.hierarchy import dendrogram, linkage
import scipy.cluster.hierarchy as shc

### **Desarrollo de dbscan**

**Columnas no utilizadas (ruido total / no sirven para modelos)**

- ID
- Name
- Photo
- Flag
- Club Logo
- Joined
- Loaned From
- Contract Valid Until
- Best Position
- Position
- Body Type
- Real Face
- Preferred Foot
- Work Rate
- Year_Joined

### **Objetivo de los clustering - ¿Que buscaremos agrupar?**

- Identificar arquetipos físicos de jugadores en función de sus capacidades atléticas. **(El objetivo se aplica a todos los los tipos de clusters.)**

**Criterios de selección**

- Variables relevantes al objetivo para segmentar.

- **No están altamente correlacionados entre sí.**

- Con varianza suficiente

- Escalables

- Interpretables

- Son útiles para identificar **arquetipos de jugadores**.

- Alta dimensionalidad

- evitar variables irrelevantes que empeoren la calidad.

**Features seleccionadas**

Siguiendo el objetivo de la agrupación, para el análisis de clustering se seleccionaron variables asociadas al rendimiento físico y capacidades atléticas del jugador, excluyendo atributos técnicos y contractuales. El objetivo es identificar perfiles físicos diferenciados dentro del conjunto de futbolistas profesionales

- Height
- Weight
- SprintSpeed
- Agility
- Balance
- ShotPower
- Jumping
- Stamina
- Strength
- Aggression

#### **Selección de features**

In [12]:
features = [
    "Height_cm",
    "Weight_kg",
    "SprintSpeed",
    "Agility",
    "Balance",
    "ShotPower",
    "Jumping",
    "Stamina",
    "Strength",
    "Aggression"
]

X = df_fifa[features]

#### **Escalado Estandarización**

In [13]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#### **Dendrograma**

Vamos a realizar un dendograma del dataset utilizando el método 'ward' para calcular distancias (es el que se suele ultilizar y viene por default). Este método minimiza la varianza dentro de un cluster y maximiza la varianza entre clusters.

Lo que **buscamos en el dendograma es la mayor distancia vertical** sin que haya una línea horizontal para hacerle un corte (representado como una linea horizontal que cruza todos los datos) y quedarnos con k clusters (donde k es el número de lineas verticales que intersectan el corte.

In [ ]:
plt.figure(figsize=(10,5))
Z = linkage(X_scaled, method="ward")
dendrogram(Z, truncate_mode="level", p=5)
plt.title("Dendrograma (Ward linkage)")
plt.xlabel("Muestras")
plt.ylabel("Distancia")
plt.show()

#### **Entrenamiento del modelo**

In [ ]:
cluster = AgglomerativeClustering(n_clusters=5,
                                  affinity='euclidean',
                                  linkage='ward'
                                  )

# Lo ajustamos con los datos
cluster.fit_predict(df_fifa)  # fit_predict hace lo mismo que fit pero devuelve el vector de etiquetas de las samples

#### **Visualización - PCA 2D** (visualizacion opcional)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8,6))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=hier_labels, palette="tab10", s=40)
plt.title("Clusters - Hierarchical Clustering (PCA 2D)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()


In [ ]:
# Ploteamos los datos en el espacio de (Ingresos,Gastos) con un color por cada uno de los 5 clusters
plt.figure(figsize=(10, 7))
plt.title("Dataset de clientes de un shopping")
plt.scatter(df_fifa[:,0], df_fifa[:,1], c=cluster.labels_, cmap='rainbow')
plt.xlabel("Ingereso anual (k$)")
plt.ylabel("Puntaje de gastos")

In [ ]:
### **Coeficiente de Silhouette**

In [ ]:
# Creamos una lista para guardar de los coeficientes de silhouette para cada valor de k
silhouette_coefficients = []

# Se necesita tener al menos 2 clusters y a los sumo N-1 (con N el numero de muestras) para obtener coeficientes de Silohuette
for k in range(2, 20):
     cluster = AgglomerativeClustering(n_clusters=k, affinity='euclidean', linkage='ward')
     cluster.fit(df_fifa)
     score = silhouette_score(df_fifa, cluster.labels_)
     silhouette_coefficients.append(score)

In [ ]:
Graficamos el puntaje de Silhouette en función de k

In [ ]:
fig, ax = plt.subplots(figsize = (24, 7))

# estas lineas son el grafico de SSE vs K
ax.scatter(range(2, 20), silhouette_coefficients)
ax.set_xticks(range(2, 20))
ax.set_xlabel("Número de clusters")
ax.set_ylabel("Promedio coeficientes de Silhouette")

In [ ]:
#comparison = df_fifa_22[["KMeans_cluster", "DBSCAN_cluster", "Hier_cluster"]]
#comparison.head()


Y un cruce de conteo:

In [ ]:
#pd.crosstab(df_fifa_22["KMeans_cluster"], df_fifa_22["Hier_cluster"])